# Chapter 6 — Build It Yourself: a BPE Tokenizer

Build the whole tokenizer from scratch — `get_stats`, `merge`, `train`, `encode`, `decode` — and
watch text compress into subword tokens. **No PyTorch here, just Python.** "✍️ Your turn",
"▶️ Run this", "▶️ Check your work". 🔤

## Step 1 — Setup: text → bytes  ▶️ Run this

Load the text and see how a string becomes a list of bytes (integers 0–255). Those 256
byte-values are our starting vocabulary.

In [ ]:
from pathlib import Path
DATA = next(p for p in [Path("../data/input.txt"), Path("data/input.txt"),
                        Path("chapters/06-tokenization/data/input.txt")] if p.exists())
text = DATA.read_text()
train_text = text[:50_000]            # train the tokenizer on this
held = text[100_000:110_000]          # measure compression on this (held-out)

print("a string as bytes:")
print(" ", list("hi 🧙".encode("utf-8")), "  <- 'h','i',' ', then 4 bytes for the emoji")
print(f"\ntext: {len(text)} chars | training on {len(train_text)} | {len(set(text))} distinct chars")

## Step 2 — Count adjacent pairs  ✍️ Your turn

Write `get_stats`: return a dict mapping each adjacent pair to how often it appears.

<details><summary>Stuck? Show the answer</summary>

```python
for pair in zip(ids, ids[1:]):
    counts[pair] = counts.get(pair, 0) + 1
```
</details>

In [ ]:
def get_stats(ids):
    counts = {}
    # ✍️ for every neighbouring pair in ids, add 1 to its count in the dict
    #    (zip pairs each item with the next; default a missing count to 0)
    return counts

print(get_stats([1, 2, 1, 2, 3]))     # expect {(1, 2): 2, (2, 1): 1, (2, 3): 1}

In [ ]:
# ▶️ Check your work
try:
    assert get_stats([1, 2, 1, 2, 3]) == {(1, 2): 2, (2, 1): 1, (2, 3): 1}, "should count each adjacent pair"
    assert get_stats([5]) == {}, "a single token has no pairs"
    print("✅ get_stats works — it counts every adjacent pair.")
except Exception as e:
    print("❌", e)

## Step 3 — Merge a pair  ✍️ Your turn

Write the two action lines of `merge`: replace every occurrence of `pair` with the new token
`idx`. The tricky index logic (the `while` and the look-ahead guard) is already there — you fill
what happens on a **match** (emit `idx`, skip 2) vs **no match** (copy one token, skip 1).

<details><summary>Stuck? Show the answer</summary>

```python
# on a match:
newids.append(idx); i += 2
# no match:
newids.append(ids[i]); i += 1
```
</details>

In [ ]:
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(ids[i]); i += 1     # ✍️ replace: on a match, emit token idx and skip BOTH
        else:
            newids.append(ids[i]); i += 1     # no match: copy this token through, advance by 1
    return newids

print(merge([1, 2, 1, 2, 3], (1, 2), 99))     # expect [99, 99, 3]

In [ ]:
# ▶️ Check your work
try:
    assert merge([1, 2, 1, 2, 3], (1, 2), 99) == [99, 99, 3], "should replace each (1,2) with 99"
    assert merge([5, 6, 7], (1, 2), 99) == [5, 6, 7], "no occurrences -> unchanged"
    assert merge([1, 2, 2], (2, 2), 99) == [1, 99], "should handle a pair of identical tokens"
    print("✅ merge works — every occurrence of the pair becomes one new token.")
except Exception as e:
    print("❌", e)

## Step 4 — Train: merge the commonest pair, repeat  ✍️ Your turn

Fill the two key lines of the loop: pick the **most common** pair, and **merge** it into the new
token `idx`. Then we train on the real text.

<details><summary>Stuck? Show the answer</summary>

```python
pair = max(stats, key=stats.get)
ids = merge(ids, pair, idx)
```
</details>

In [ ]:
def train(text, vocab_size):
    num_merges = vocab_size - 256
    ids = list(text.encode("utf-8"))
    merges = {}
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for i in range(num_merges):
        stats = get_stats(ids)
        idx = 256 + i
        pair = (0, 0)         # ✍️ replace: the MOST COMMON pair (the one with the highest count)
        ids = ids            # ✍️ replace: apply that merge to ids (folding the pair into token idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    return merges, vocab

merges, vocab = train(train_text, 512)
print(f"learned {len(merges)} merges; vocab size {len(vocab)}")
print("first 4 merges:", [(bytes(vocab[a]), bytes(vocab[b])) for (a, b) in list(merges)[:4]])

In [ ]:
# ▶️ Check your work
try:
    m, v = train("ab ab ab ab", 257)            # commonest pair is ('a','b')
    assert len(m) == 1 and v[256] == b"ab", "the one merge should be 'a'+'b' -> b'ab'"
    assert len(merges) == 256 and len(vocab) == 512, "the real tokenizer should have 256 merges / vocab 512"
    print(f"✅ train works — {len(merges)} merges learned, vocab {len(vocab)}.")
except Exception as e:
    print("❌", e)

## Step 5 — Encode new text  ✍️ Your turn

Re-apply the learned merges to new text, **earliest-learned first**. Fill the pair-selection (the
`min` trick) and the merge. The `break` is given.

<details><summary>Stuck? Show the answer</summary>

```python
pair = min(stats, key=lambda p: merges.get(p, float("inf")))
...
ids = merge(ids, pair, merges[pair])
```
The `min` picks the present pair with the smallest merge id (earliest learned); `float("inf")`
sends non-merges to the back so the `break` fires when nothing is merge-able.
</details>

In [ ]:
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = (-1, -1)  # ✍️ replace: the present pair with the SMALLEST merge id (earliest learned;
                         #    send pairs that were never merged to the back so the break can fire)
        if pair not in merges:
            break        # nothing learned is present -> stop
        ids = ids        # ✍️ replace: apply that merge to ids
    return ids

print(len(held.encode("utf-8")), "bytes ->", len(encode(held)), "tokens")

In [ ]:
# ▶️ Check your work
try:
    ids = encode(held)
    assert all(isinstance(t, int) for t in ids)
    assert len(ids) < len(held.encode("utf-8")), "encode should make FEWER tokens than bytes (apply the merges)"
    assert any(t >= 256 for t in ids), "encode should actually use the merged tokens (ids >= 256)"
    assert len(ids) == 4971, "wrong token COUNT — apply merges in LEARNED order (the min-trick), not by most-frequent"
    print(f"✅ encode works — it compressed the bytes using your learned merges, in the right order.")
except Exception as e:
    print("❌", e)

## Step 6 — Decode, and prove the round-trip 🔑  ✍️ Your turn

Decode is the easy mirror: glue each token's bytes (from `vocab`) and decode the UTF-8. Then the
golden test — `decode(encode(text))` must equal the original, even for emoji and accents.

<details><summary>Stuck? Show the answer</summary>

```python
text_bytes = b"".join(vocab[idx] for idx in ids)
```
</details>

In [ ]:
def decode(ids):
    text_bytes = b""     # ✍️ replace: look up each token's bytes in vocab and join them in order
    return text_bytes.decode("utf-8", errors="replace")

s = "First Citizen — wherefore? 🧙 café"
print("round-trips:", decode(encode(s)) == s)

In [ ]:
# ▶️ Check your work
try:
    for s in ["First Citizen — wherefore? 🧙 café", "", "a", "the quick brown fox"]:
        assert decode(encode(s)) == s, f"round-trip failed on {s!r}"
    hb, ht = len(held.encode("utf-8")), len(encode(held))
    print(f"✅ round-trip works for all of them! compression on held-out: {hb} -> {ht} ({hb/ht:.2f}x)")
except Exception as e:
    print("❌", e)

## 🎉 You built a tokenizer.

`get_stats` → `merge` → `train` → `encode` → `decode`: the exact algorithm behind GPT-2 and
GPT-4, just smaller. Next: the [exercises](../exercises/) (plot the vocab↔compression curve, wire
up GPT-2's regex splitting) and the [mini-project](../project/), *Train Your Own Tokenizer*. 👋